# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # This is an object, not a dict/list

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we list all record sets in the dataset. Each entity is referenced by its `@id` as per Croissant best practices.

In [ ]:
# List all record sets by their @id and name

record_sets = dataset.record_sets()
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}, name: {rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, data type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.
**Below we will extract the main protein analysis table, commonly named like `mass_spec_analysis` or similar. Please choose the desired record set `@id` from the previous output.**

In [ ]:
# For the sake of demonstration, we'll extract all record sets.

dataframes = {}
record_sets = dataset.record_sets()

for rs in record_sets:
    # Use the record set @id as the key
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df

# List extracted DataFrame columns of the first record set
if record_sets:
    main_rs_id = record_sets[0].id
    print(f"Columns for record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We'll use the main record set and demonstrate filtering and normalization with a numeric field, referenced by field `@id`.

In [ ]:
import numpy as np

# Choose main record set for EDA
record_set_id = main_rs_id
df = dataframes[record_set_id]
print(f"Working with record set: {record_set_id}")

# List available fields for convenience
print("Available columns:", df.columns.tolist())

# Suppose the numeric field is named by its @id, e.g., 'coverage_percentage' or similar.
# You should replace <numeric_field_id> below with the actual @id from your earlier field output. For demonstration, let's try to pick a suitable name:

# For the sake of example, let's use 'coverage_percentage' if present, otherwise choose the first numeric-like field.
def pick_numeric_field(df):
    # Try to pick the best numeric field likely to be present
    for candidate in ['coverage_percentage', 'molecular_weight', 'num_peptides', 'mw', 'pi', 'normalized_abundance']:
        if candidate in df.columns:
            return candidate
    # Fallback: pick first float/integer column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            return col
    return df.columns[0]

numeric_field = pick_numeric_field(df)
print(f"Selected numeric field: {numeric_field}")

threshold = 10

# Filtering based on the numeric field
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a possible categorical field (e.g., 'sample_id', 'condition', 'group_label', etc.)
group_candidates = ['sample_id', 'condition', 'group', 'experiment']
group_field = None
for candidate in group_candidates:
    if candidate in df.columns:
        group_field = candidate
        break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field found in columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If a group field is available, show group comparison
if group_field is not None:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides rich protein analysis data with fields such as coverage percentage, peptide counts, molecular weight, and more.
- Initial EDA demonstrates how to filter, normalize, and group by record set fields using Croissant `@id` references.
- Visualizations allow inspection of data distributions and potential group-wise trends.

Further analysis can be performed depending on the scientific question of interest, such as comparing protein abundances across conditions or exploring relationships between biophysical properties.